# Classificação da Qualidade da Água a partir de Dados Ambientais
**TCC 1 — Felipe Rosa Ferreira — PUCRS**

Este notebook implementa e avalia dois modelos de Machine Learning para classificação de potabilidade da água:
- **Naive Bayes** (GaussianNB)
- **Regressão Logística**

Dataset: [Water Quality / Water Potability — Kaggle](https://www.kaggle.com/datasets/adityakadiwal/water-potability)


## Passo 1 — Instalar bibliotecas necessárias

In [ ]:
# Instala a biblioteca de balanceamento de classes (SMOTE)
# As demais (pandas, sklearn, matplotlib, seaborn) já vêm no Colab
!pip install imbalanced-learn -q

## Passo 2 — Importar bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE

print('Bibliotecas importadas com sucesso!')

## Passo 3 — Fazer upload do dataset

Execute a célula abaixo e selecione o arquivo **datasetcorretoPortugues.xlsx** do seu computador.

In [ ]:
from google.colab import files
uploaded = files.upload()  # abre janela para selecionar o arquivo

## Passo 4 — Carregar e explorar o dataset

In [ ]:
# Carrega o arquivo que você acabou de fazer upload
nome_arquivo = list(uploaded.keys())[0]
df = pd.read_excel(nome_arquivo)

print(f'Dimensões do dataset: {df.shape[0]} amostras | {df.shape[1]} colunas')
print(f'\nColunas: {df.columns.tolist()}')
print(f'\nDistribuição da variável alvo (Potability):')
print(df['Potability'].value_counts())
print(f'\nValores ausentes por coluna:')
print(df.isnull().sum())

In [ ]:
# Visualizar as primeiras linhas do dataset
df.head()

In [ ]:
# Estatísticas descritivas
df.describe().round(2)

## Passo 5 — Pré-processamento

In [ ]:
# 1. Preencher valores ausentes com a média de cada coluna
df.fillna(df.mean(numeric_only=True), inplace=True)

# 2. Separar features (X) do alvo (y)
X = df.drop('Potability', axis=1)   # 9 colunas: pH, Hardness, etc.
y = df['Potability']                # 0 = Não Potável | 1 = Potável

print('Valores ausentes após preenchimento:', df.isnull().sum().sum())
print(f'Features: {X.shape[1]} | Amostras: {X.shape[0]}')

## Passo 6 — Divisão Treino/Teste (80% / 20%)

In [ ]:
# Divide o dataset: 80% para treino, 20% para teste
# stratify=y garante a mesma proporção de classes nos dois conjuntos
# random_state=42 garante que a divisão seja sempre igual (reprodutível)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')
print(f'\nDistribuição no treino: {y_train.value_counts().to_dict()}')
print(f'Distribuição no teste:  {y_test.value_counts().to_dict()}')

## Passo 7 — Balanceamento com SMOTE

O dataset é desbalanceado (mais amostras "Não Potável" que "Potável").
O SMOTE cria amostras sintéticas da classe minoritária **somente no treino**.

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Antes do SMOTE: {dict(y_train.value_counts())}')
print(f'Após o SMOTE:   {dict(zip(*np.unique(y_train_sm, return_counts=True)))}')

## Passo 8 — Normalização dos dados

In [ ]:
# Normaliza os dados para média=0 e desvio padrão=1
# Importante especialmente para a Regressão Logística
# fit_transform no treino, apenas transform no teste (para não vazar informação)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc  = scaler.transform(X_test)

print('Normalização aplicada com sucesso!')

## Passo 9 — Treinar e Avaliar os Modelos

In [ ]:
# ── NAIVE BAYES ──────────────────────────────────────────────────────────────
nb = GaussianNB()
nb.fit(X_train_sc, y_train_sm)          # treina
y_pred_nb = nb.predict(X_test_sc)       # prevê
y_prob_nb = nb.predict_proba(X_test_sc)[:, 1]  # probabilidade (para AUC-ROC)

print('=' * 55)
print('  NAIVE BAYES')
print('=' * 55)
print(classification_report(y_test, y_pred_nb,
      target_names=['Não Potável (0)', 'Potável (1)']))
print(f'  AUC-ROC: {roc_auc_score(y_test, y_prob_nb):.4f}')

In [ ]:
# ── REGRESSÃO LOGÍSTICA ───────────────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train_sm)          # treina
y_pred_lr = lr.predict(X_test_sc)       # prevê
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]

print('=' * 55)
print('  REGRESSÃO LOGÍSTICA')
print('=' * 55)
print(classification_report(y_test, y_pred_lr,
      target_names=['Não Potável (0)', 'Potável (1)']))
print(f'  AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.4f}')

## Passo 10 — Tabela Comparativa Final

In [ ]:
dados = {
    'Modelo': ['Naive Bayes', 'Regressão Logística'],
    'Acurácia': [
        round(accuracy_score(y_test, y_pred_nb), 4),
        round(accuracy_score(y_test, y_pred_lr), 4)
    ],
    'Precisão': [
        round(precision_score(y_test, y_pred_nb), 4),
        round(precision_score(y_test, y_pred_lr), 4)
    ],
    'Recall': [
        round(recall_score(y_test, y_pred_nb), 4),
        round(recall_score(y_test, y_pred_lr), 4)
    ],
    'F1-Score': [
        round(f1_score(y_test, y_pred_nb), 4),
        round(f1_score(y_test, y_pred_lr), 4)
    ],
    'AUC-ROC': [
        round(roc_auc_score(y_test, y_prob_nb), 4),
        round(roc_auc_score(y_test, y_prob_lr), 4)
    ]
}

tabela = pd.DataFrame(dados)
tabela

## Passo 11 — Visualizações: Matrizes de Confusão e Curvas ROC

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    'Naive Bayes vs Regressão Logística — Water Potability\n(com SMOTE)',
    fontsize=14, fontweight='bold'
)

labels = ['Não Potável', 'Potável']
cores  = ['#4A90D9', '#C0392B']

modelos = [
    ('Naive Bayes',         y_pred_nb, y_prob_nb, cores[0]),
    ('Regressão Logística', y_pred_lr, y_prob_lr, cores[1])
]

for i, (nome, yp, yb, cor) in enumerate(modelos):

    # Matriz de confusão
    cm = confusion_matrix(y_test, yp)
    ax = axes[0][i]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels, annot_kws={'size': 13})
    acc = accuracy_score(y_test, yp)
    f1  = f1_score(y_test, yp)
    ax.set_title(f'Matriz de Confusão — {nome}\nAcc={acc:.3f} | F1={f1:.3f}',
                 fontweight='bold')
    ax.set_ylabel('Valor Real')
    ax.set_xlabel('Valor Previsto')

    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, yb)
    auc = roc_auc_score(y_test, yb)
    ax2 = axes[1][i]
    ax2.plot(fpr, tpr, color=cor, lw=2.5, label=f'AUC = {auc:.4f}')
    ax2.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax2.fill_between(fpr, tpr, alpha=0.08, color=cor)
    ax2.set_title(f'Curva ROC — {nome}', fontweight='bold')
    ax2.set_xlabel('Taxa de Falsos Positivos (FPR)')
    ax2.set_ylabel('Taxa de Verdadeiros Positivos (TPR)')
    ax2.legend(loc='lower right', fontsize=11)
    ax2.set_xlim([0, 1])
    ax2.set_ylim([0, 1.02])
    ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('resultados_modelos.png', dpi=200, bbox_inches='tight')
plt.show()
print('Gráfico salvo como resultados_modelos.png')

## Passo 12 — Baixar o gráfico gerado

In [ ]:
# Faz o download do gráfico para o seu computador
from google.colab import files
files.download('resultados_modelos.png')

---
## Conclusão

Os resultados dos modelos baseline (Naive Bayes e Regressão Logística) com AUC-ROC em torno de 0.59 indicam que esses modelos lineares e probabilísticos simples não são suficientes para capturar a complexidade dos padrões de potabilidade da água.

Isso **justifica** a utilização de modelos mais sofisticados no TCC 2, em especial o **MLP (Multilayer Perceptron)**, capaz de aprender relações não-lineares entre os parâmetros físico-químicos da água e sua potabilidade.
